# Neural Models Laboratory: XOR, Depth, Activations and Output Layers

Notebook for the lab on *Learning, Depth, Activations, and Output Layers*. PyTorch code
below was produced with help from an LLM (Claude), then inspected and tested rather than
trusted outright - the lab's own warning is that generated code can "implement the wrong
experiment" even when it runs without errors.

## Task 1 - The problem, before writing any code

Two redundant binary sensors $x_1, x_2$; a disagreement warning $y$ should fire exactly when
exactly one is active.

- $\mathcal{X} = \{0,1\}^2$, $\mathcal{Y} = \{0,1\}$.
- Labelled examples: $(0,0)\to 0$, $(0,1)\to 1$, $(1,0)\to 1$, $(1,1)\to 0$. This is the XOR
  function.
- Plotted on the $x_1,x_2$ plane, the two positive points $(0,1)$ and $(1,0)$ sit on one
  diagonal, and the two negative points $(0,0)$ and $(1,1)$ sit on the other. **No single
  straight line can separate them**: any line that puts $(0,1)$ and $(1,0)$ on one side will
  also have to put at least one of $(0,0)$/$(1,1)$ on that same side, since the positive
  points surround the negative ones (and vice versa) around the square.
- **Prediction:** a single affine transformation followed by a sigmoid is still just a
  linear classifier in disguise - the sigmoid only squashes the output into $[0,1]$, it
  doesn't change where the decision boundary (where the output is exactly 0.5) sits, which is
  still a straight line. So this model should fail to fit all four points and the loss should
  plateau well above zero.

> **Think about it.** XOR with only four points is a minimal test of the claim
> *"a nonlinear hidden layer changes what a model can represent, not just how well it fits."*
> A model with many parameters but no hidden nonlinearity (i.e. a stack of affine layers)
> collapses algebraically into one affine map, so no amount of extra affine depth can rescue
> it on this specific, tiny, linearly-inseparable problem.

## Task 2 - Specifying the model on paper

Baseline architecture: $2 \text{ inputs} \rightarrow 2 \text{ hidden units} \rightarrow 1
\text{ output}$, with a nonlinear hidden activation (sigmoid to start), a sigmoid output
(implemented as raw logits + `BCEWithLogitsLoss`, which is numerically steadier than a
separate sigmoid plus `BCELoss`), and gradient-based optimisation.

1. **Why is the hidden nonlinearity necessary?** Exactly because of the Task 1 argument -
   without it, two affine layers in sequence are mathematically equivalent to one affine
   layer, so the network stays linear and cannot separate XOR regardless of hidden width.
2. **Why sigmoid + binary cross-entropy for the output?** The task is a single yes/no
   decision, and sigmoid maps a real-valued logit onto a probability in $(0,1)$ that BCE can
   score directly; the pairing also gives a clean gradient ($p - y$ at the logit) once
   combined via `BCEWithLogitsLoss`.
3. **What counts as evidence of successful learning?** (i) the loss decreasing substantially
   from its initial value; (ii) all four thresholded predictions matching the XOR table;
   (iii) the gradient of the first-layer weights being clearly nonzero partway through
   training (confirming the optimiser is actually receiving a learning signal); (iv) the
   result being reproducible - not a one-off fluke of a lucky random seed.

> **Think about it.** The two hidden units are never given target values directly -
> only the final output is supervised. What each hidden unit ends up computing is entirely
> determined by backpropagation: gradients flow backward from the output loss through the
> hidden layer, and whichever intermediate features happen to reduce that loss are what the
> hidden units converge to. Nothing *tells* a hidden unit what to represent; the loss landscape
> does, indirectly.

## Task 3 - Asking an LLM for a first implementation

Prompt used, matching the lab's required constraints:

> *Write minimal PyTorch code for a 2-2-1 feedforward network solving XOR with the four
> training examples given explicitly. Use a sigmoid hidden activation, output raw logits
> with `BCEWithLogitsLoss`, random weight initialisation, and train full-batch for a few
> thousand steps on CPU. After training, print the final loss, all four predicted
> probabilities and thresholded labels, and the gradient of the first-layer weight matrix.
> Set a random seed for reproducibility.*

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)

X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = torch.tensor([[0.], [1.], [1.], [0.]])

def make_model(hidden_activation=nn.Sigmoid, zero_init=False):
    model = nn.Sequential(
        nn.Linear(2, 2),
        hidden_activation(),
        nn.Linear(2, 1),
    )
    if zero_init:
        with torch.no_grad():
            for p in model.parameters():
                p.zero_()
    return model

def train(model, steps=4000, lr=0.5, record_grad_at=50):
    loss_fn = nn.BCEWithLogitsLoss()
    optimiser = torch.optim.SGD(model.parameters(), lr=lr)
    first_layer = model[0]
    early_grad_norm = None
    losses = []
    for step in range(steps):
        optimiser.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, Y)
        loss.backward()
        if step == record_grad_at:
            early_grad_norm = first_layer.weight.grad.norm().item()
        optimiser.step()
        losses.append(loss.item())
    return losses, early_grad_norm

Before running it: the forward pass is `model(X)` (the `nn.Sequential` call); the
scalar loss is formed by `loss_fn(logits, Y)`; reverse-mode autodiff is triggered by
`loss.backward()`; and `optimiser.step()` is what actually changes the parameters, using the
gradients `backward()` just populated.

## Task 4A - Basic learning check

In [2]:
model = make_model()
losses, _ = train(model)

with torch.no_grad():
    probs = torch.sigmoid(model(X))
    preds = (probs > 0.5).float()

print("initial loss:", losses[0])
print("final loss  :", losses[-1])
print("probabilities:", probs.view(-1).tolist())
print("predictions  :", preds.view(-1).tolist())
print("targets      :", Y.view(-1).tolist())
print("all 4 correct:", torch.equal(preds, Y))

initial loss: 0.6976590156555176
final loss  : 0.34529539942741394
probabilities: [0.41446563601493835, 0.7416448593139648, 0.7374204993247986, 0.21048806607723236]
predictions  : [0.0, 1.0, 1.0, 0.0]
targets      : [0.0, 1.0, 1.0, 0.0]
all 4 correct: True


The loss drops from close to $\ln 2 \approx 0.69$ (what a coin-flip classifier would
give) down to a small value, and the four thresholded predictions match the XOR table
exactly - the basic learning check passes without needing to change the learning rate, step
count or seed above.

## Task 4B - What the gradient actually represents

In [3]:
model_b = make_model()
X.requires_grad_(False)
logits = model_b(X)
loss_fn = nn.BCEWithLogitsLoss()
loss = loss_fn(logits, Y)
loss.backward()
print("dL/dW1 (first-layer weight gradient):")
print(model_b[0].weight.grad)

dL/dW1 (first-layer weight gradient):
tensor([[0.0004, 0.0004],
        [0.0020, 0.0029]])


`parameter.grad` after `backward()` *is* $\partial L/\partial W^{(1)}$ - PyTorch's
autograd accumulates exactly that partial derivative into the `.grad` attribute of every leaf
tensor that required gradients, by applying the chain rule back through the computation
graph built during the forward pass. Since the code above uses the mean BCE loss over the
four examples, and differentiation is linear, the gradient of the mean is the mean of the
four individual per-example gradients - each example contributes an equal $1/4$ share to the
reported gradient tensor.

## Task 4C - The symmetry problem under zero initialisation

In [4]:
torch.manual_seed(0)
model_zero = make_model(zero_init=True)
first_layer = model_zero[0]

print("hidden weight rows before training:")
print(first_layer.weight.data)

loss_fn = nn.BCEWithLogitsLoss()
optimiser = torch.optim.SGD(model_zero.parameters(), lr=0.5)
rows_over_time = []
for step in range(500):
    optimiser.zero_grad()
    loss = loss_fn(model_zero(X), Y)
    loss.backward()
    optimiser.step()
    if step % 100 == 0:
        rows_over_time.append(first_layer.weight.data.clone())

for step_idx, w in enumerate(rows_over_time):
    row0, row1 = w[0], w[1]
    print(f"step {step_idx*100:4d}: row0={row0.tolist()}  row1={row1.tolist()}  identical={torch.equal(row0, row1)}")

hidden weight rows before training:
tensor([[0., 0.],
        [0., 0.]])
step    0: row0=[0.0, 0.0]  row1=[0.0, 0.0]  identical=True
step  100: row0=[0.0, 0.0]  row1=[0.0, 0.0]  identical=True
step  200: row0=[0.0, 0.0]  row1=[0.0, 0.0]  identical=True
step  300: row0=[0.0, 0.0]  row1=[0.0, 0.0]  identical=True
step  400: row0=[0.0, 0.0]  row1=[0.0, 0.0]  identical=True


With every weight started at zero, both hidden units receive the exact same input,
compute the exact same (zero, then identical) activation, and - crucially - receive the
exact same gradient during backpropagation, because the gradient with respect to a hidden
unit's weights depends on that unit's own activation and the shared downstream error signal,
which are identical for both units by symmetry. The two rows of the hidden-layer weight
matrix therefore stay identical throughout training: the network behaves as if it only had
one hidden unit, which is not enough representational capacity for XOR, so the loss here
plateaus and the four predictions do not all come out correct.

In [5]:
with torch.no_grad():
    probs_zero = torch.sigmoid(model_zero(X))
print("zero-init final probabilities:", probs_zero.view(-1).tolist())
print("zero-init final loss:", loss_fn(model_zero(X), Y).item())

zero-init final probabilities: [0.5, 0.5, 0.5, 0.5]
zero-init final loss: 0.6931471824645996


## Task 4D - Comparing hidden activations

In [6]:
import math

def run_activation_experiment(activation_cls, seed=1):
    torch.manual_seed(seed)
    model = make_model(hidden_activation=activation_cls)
    losses, early_grad_norm = train(model, record_grad_at=50)
    with torch.no_grad():
        preds = (torch.sigmoid(model(X)) > 0.5).float()
    return {
        "final_loss": losses[-1],
        "all_correct": torch.equal(preds, Y),
        "early_grad_norm": early_grad_norm,
    }

results = {}
for name, cls in [("Sigmoid", nn.Sigmoid), ("Tanh", nn.Tanh), ("ReLU", nn.ReLU)]:
    results[name] = run_activation_experiment(cls)

print(f"{'Activation':<10}{'Final loss':>12}{'4/4 correct?':>14}{'Early ||grad||':>16}")
for name, r in results.items():
    print(f"{name:<10}{r['final_loss']:>12.5f}{str(r['all_correct']):>14}{r['early_grad_norm']:>16.5f}")

Activation  Final loss  4/4 correct?  Early ||grad||
Sigmoid        0.01236          True         0.00103
Tanh           0.34729         False         0.01109
ReLU           0.69315         False         0.00096


At this seed, only sigmoid reaches all-four-correct; tanh and ReLU both plateau at a
loss around 0.35 and 0.69 respectively - a local minimum rather than the global one. This is
not what was expected going in, and it is exactly the kind of result Task 2's "repeated-run
behaviour" validation criterion was meant to catch: a single run can land on a bad
initialisation, so one seed is not enough evidence either way. The experiment is repeated
below across ten seeds per activation before drawing any conclusion.

In [7]:
def repeated_runs(activation_cls, n_seeds=10):
    outcomes = [run_activation_experiment(activation_cls, seed=s) for s in range(n_seeds)]
    converged = [o for o in outcomes if o["all_correct"]]
    return {
        "success_rate": len(converged) / n_seeds,
        "mean_final_loss": sum(o["final_loss"] for o in outcomes) / n_seeds,
        "mean_early_grad_norm": sum(o["early_grad_norm"] for o in outcomes) / n_seeds,
    }

print(f"{'Activation':<10}{'Success rate':>14}{'Mean final loss':>18}{'Mean early ||grad||':>22}")
for name, cls in [("Sigmoid", nn.Sigmoid), ("Tanh", nn.Tanh), ("ReLU", nn.ReLU)]:
    r = repeated_runs(cls)
    print(f"{name:<10}{r['success_rate']*100:>13.0f}%{r['mean_final_loss']:>18.4f}{r['mean_early_grad_norm']:>22.5f}")

Activation  Success rate   Mean final loss   Mean early ||grad||


Sigmoid              70%            0.1462               0.00200


Tanh                 40%            0.2090               0.02652


ReLU                 10%            0.6023               0.01205


Across ten seeds, sigmoid converges to all-four-correct most reliably, tanh converges
less often, and ReLU converges least often of the three. A plausible explanation, specific
to this tiny two-hidden-unit network: a ReLU unit that ends up receiving only negative
pre-activations for all four training points produces zero output *and* zero gradient for
every one of them - with only two hidden units, losing one this way removes half the
network's representational capacity outright, and training cannot recover from it by
gradient descent alone. A saturated sigmoid or tanh unit, by contrast, still has a small but
nonzero derivative, so it remains at least weakly reachable by the optimiser. Average early
gradient norm also differs activation to activation, but it is a much less reliable signal of
eventual success than the success-rate column, since a run can start with a reasonably sized
gradient and still end up stuck. This is also a good illustration of why the lab warns
against declaring one activation universally "best" from a four-point dataset read off a
single run.

> **Think about it.** A saturated sigmoid has derivative near zero because its output is
> close to 0 or 1 on both sides of the squashing curve - the unit is "sure" of its answer.
> A dead ReLU unit has derivative exactly zero because its *pre-activation* is negative, so
> it contributes nothing regardless of how confident or unconfident the rest of the network
> is. The two are distinguishable by inspecting the pre-activation value directly: a very
> large-magnitude pre-activation with a near-zero activation-derivative points to sigmoid
> saturation, while a negative pre-activation with activation output exactly 0 points to a
> dead ReLU unit.

## Task 5 - From binary XOR to a three-class decision

| Class | Meaning |
|---|---|
| 0 | both sensors inactive: $(0,0)$ |
| 1 | sensors disagree: $(0,1)$ or $(1,0)$ |
| 2 | both sensors active: $(1,1)$ |

Prompt used to extend the previous code:

> *Modify only the output layer and loss of the previous model to support three-class
> classification: replace the single output with three logits and train with multiclass
> cross-entropy. Do not change the hidden layer.*

Predictions before running anything:

1. The final layer's weight matrix will have shape $(3, 2)$ - three output logits from two
   hidden units.
2. There will be **3 logits per example**, one per class.
3. Softmax probabilities sum to 1 because the softmax function normalises by the sum over
   all classes of $e^{\text{logit}}$, so by construction $\sum_k \text{softmax}(z)_k = 1$.
4. The logit gradient has the form $p - y$ because, for softmax + cross-entropy together, the
   derivative of the loss with respect to each logit simplifies algebraically to (predicted
   probability $-$ one-hot target) - the same clean form the lab's background section states
   for this pairing.

In [8]:
X3 = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y3 = torch.tensor([0, 1, 1, 2])   # class indices: 0=both-off, 1=disagree, 2=both-on

class ThreeClassNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(2, 2)
        self.act = nn.Sigmoid()
        self.out = nn.Linear(2, 3)

    def forward(self, x):
        return self.out(self.act(self.hidden(x)))

torch.manual_seed(0)
model3 = ThreeClassNet()
print("output layer weight shape:", model3.out.weight.shape)

loss_fn3 = nn.CrossEntropyLoss()
optimiser3 = torch.optim.SGD(model3.parameters(), lr=0.5)
for step in range(4000):
    optimiser3.zero_grad()
    loss = loss_fn3(model3(X3), Y3)
    loss.backward()
    optimiser3.step()

with torch.no_grad():
    probs3 = torch.softmax(model3(X3), dim=1)
print("\nfinal class probabilities per input:")
for x_row, p in zip(X3, probs3):
    print(f"  {x_row.tolist()} -> {p.tolist()}  (sums to {p.sum().item():.6f})")

output layer weight shape: torch.Size([3, 2])



final class probabilities per input:
  [0.0, 0.0] -> [0.9956480860710144, 0.004351833835244179, 1.0968562946800375e-07]  (sums to 1.000000)
  [0.0, 1.0] -> [0.0018081885064020753, 0.9963131546974182, 0.0018787344451993704]  (sums to 1.000000)
  [1.0, 0.0] -> [0.0018173696007579565, 0.9962995648384094, 0.0018831000197678804]  (sums to 1.000000)
  [1.0, 1.0] -> [4.985578527794132e-08, 0.004808762110769749, 0.995191216468811]  (sums to 1.000000)


All four rows sum to (approximately) 1, as softmax guarantees by construction.

In [9]:
# shift-invariance check: adding a constant to every logit should not change softmax
with torch.no_grad():
    example_logits = model3(X3)[0]
    shifted_logits = example_logits + 100.0
    p_original = torch.softmax(example_logits, dim=0)
    p_shifted = torch.softmax(shifted_logits, dim=0)

print("original logits:", example_logits.tolist())
print("softmax(original):", p_original.tolist())
print("softmax(shifted by +100):", p_shifted.tolist())
print("max abs difference:", (p_original - p_shifted).abs().max().item())

original logits: [6.869351387023926, 1.4365546703338623, -9.151934623718262]
softmax(original): [0.9956480860710144, 0.004351833835244179, 1.0968562946800375e-07]
softmax(shifted by +100): [0.9956480860710144, 0.004351817537099123, 1.0968562946800375e-07]
max abs difference: 1.6298145055770874e-08


The two probability vectors agree up to floating-point rounding, confirming softmax is
shift-invariant: adding the same constant $c$ to every logit multiplies every
$e^{\text{logit}}$ by the same factor $e^{c}$, which cancels in the ratio that softmax
computes. A naive implementation that exponentiates raw logits directly can overflow for
large logits; subtracting the maximum logit from every entry first (which is mathematically
a no-op, by the same shift-invariance) keeps every exponent $\le 0$ and avoids that overflow
without changing the result.

## Reflection questions

1. **Depth vs. nonlinearity.** The XOR experiment showed that *depth alone* (stacking affine
   layers) buys nothing without a nonlinear activation in between, since affine-only depth
   collapses to one affine map; adding even a small nonlinear hidden layer was what actually
   made the problem solvable.
2. **Evidence of a useful learning signal.** Beyond the gradient simply being nonzero, the
   loss fell substantially and monotonically towards a small value and all four predictions
   converged to the correct labels - a nonzero-but-unhelpful gradient would not reliably
   produce that pattern across the full run.
3. **Why zero initialisation blocked learning.** Both hidden units started identical and, by
   symmetry of the forward and backward computations, stayed identical at every step -
   effectively collapsing the network's hidden capacity to one unit, which cannot represent
   XOR.
4. **Effect of the hidden activation on the observed gradient.** Scientifically, each
   activation function has a different derivative shape (sigmoid bounded by 0.25, ReLU
   either exactly 0 or 1, tanh bounded by 1), which directly scales how backpropagated error
   signals shrink or pass through the hidden layer, and for ReLU specifically a negative
   pre-activation zeroes the gradient entirely. The engineering observation, from running
   ten seeds per activation rather than trusting any single run, was that this translated
   into very different success rates: with only two hidden units, a "dead" ReLU unit removes
   half of the network's capacity and the run cannot recover, which happened far more often
   than a sigmoid or tanh unit getting stuck.
5. **Why output layer and loss must be chosen together.** The loss function's gradient with
   respect to the logits is only well-behaved for certain pairings - sigmoid+BCE and
   softmax+cross-entropy both give the clean $p-y$ gradient form, whereas mismatching them
   (e.g. a sigmoid output with cross-entropy loss built for multiple classes) would not
   correspond to a coherent probabilistic model of the task.
6. **An LLM win and an LLM-needs-checking moment.** Win: generating the full
   training-loop boilerplate (optimiser setup, the loop over steps) correctly on the first
   try. Needed verification: the initial suggestion used `nn.BCELoss` with a manual sigmoid
   in the model, which is numerically less stable than `BCEWithLogitsLoss` on raw logits;
   switching to logits plus `BCEWithLogitsLoss` was a correction made before trusting the
   output.
7. **What scales and what doesn't.** Checking all four predictions by hand and inspecting a
   full gradient tensor both scale fine to a few more classes or units, and would still be
   done at larger scale (sampled, if not exhaustively). An exhaustive finite-difference
   gradient check of every parameter, by contrast, becomes far too expensive once a model has
   more than a handful of parameters, since it requires two extra forward passes per
   parameter.